In [ ]:
#| echo: false
import os

import retentioneering as rete
from IPython.display import HTML

# Rendered by Quarto, slides have no Python kernel behind them, so every widget
# is baked in as a self-contained static page (render_static) instead of a live
# anywidget, with its settings sidebar closed to leave the slide to the
# chart. Run in Jupyter, the widgets stay live.
if "QUARTO_DOCUMENT_PATH" in os.environ:
    from retentioneering.widgets._base import RetentioneeringWidget

    # A widget can ask for its own height on a slide: `widget.slide_height = …`
    # (the widget's layout follows its `height`, the iframe follows the page).
    SLIDE_WIDGET_HEIGHT = 470

    def _static_bundle(self, include=None, exclude=None, **kwargs):
        height = getattr(self, "slide_height", None)
        if height:
            self.height = height
        return {
            "text/html": self.render_static(
                height=height or SLIDE_WIDGET_HEIGHT, sidebar_open=False
            ).data
        }

    RetentioneeringWidget._repr_mimebundle_ = _static_bundle

## About us

### Vladimir Kukushkin

![](img/vladimir-kukushkin.jpg)

- 13 years in product analytics
- ex. Member of Techincal Staff @ Perplexity
- Principal AI Quality Analyst @ Finom
- Co-founder of Retentioneering

### Maxim Godzi

![](img/maxim-godzi.jpg)

- PhD in XXX
- NN years in product analytics
- ex. Principal Data Science Manager @ Microsoft
- Co-founder of Retentioneering

# Part 1: Motivating example

## How does a team adopt a task tracker?

- **Taskboard**, a task tracker for teams: 20,013 users in 14,000 workspaces over eight weeks, 132,547 visits, 1,045,248 events of 31 types.
- **Free for two, paid for teams**: a workspace is free for up to two members; a third one needs the Team plan, which the workspace owner buys. Reports, automations and integrations are free for everyone.
- So the business needs owners to bring a team in, and to grow it past two.


In [ ]:
#| echo: false
#| output: false
import pandas as pd

# A static snapshot of taskboard.load_taskboard(): 14,000 workspaces, seed 1.
taskboard = rete.Eventstream(
    pd.read_csv("taskboard.csv.gz", parse_dates=["timestamp"]),
    schema={
        "path_cols": ["user_id", "session_id"],
        "segment_cols": ["channel", "platform", "ab_group"],
        "custom_cols": ["workspace_id"],
    },
)


In [ ]:
#| echo: false
from toy import app_mockups

HTML(app_mockups())


## Events and the happy path

| Getting started | Tasks | Discussion | Team | Power features | Billing |
|---|---|---|---|---|---|
| `signup` | `open_board` | `comment` | `invite_member` | `view_report` | `view_pricing` |
| `tour_step` | `create_task` | `mention` | `view_invite_status` | `export` | `select_plan` |
| `template_view` | `edit_task` | `view_notifications` | `resend_invite` | `create_automation` | `start_checkout` |
| `create_project` | `complete_task` |  | `accept_invite` | `connect_integration` | `enter_payment` |
|  | `assign_task` |  | `teammate_joined` |  | `payment_error` |
|  | `set_due_date` |  |  |  | `support_ticket` |
|  | `add_dependency` |  |  |  | `upgrade` |
|  | `view_timeline` |  |  |  |  |

**Owner's happy path:** `signup` → `create_project` → `invite_member` → `teammate_joined` → `invite_member` → `view_pricing` → `upgrade`


## Every path is unique

In [ ]:
#| echo: false
from toy import (
    branching_figure,
    distinct_colors,
    first_visits_figure,
    stage_graph,
    taskboard_colors,
    visit_paths_figure,
)

USERS = ["user_00118", "user_00027", "user_00029", "user_00016"]
# The trees: a random 1,000 of the 20,013 users is enough to see the point.
sample = taskboard.sample_paths(n=1000, random_state=7)
event_colors_31 = distinct_colors(taskboard.df["event"])
HTML(
    first_visits_figure(taskboard, USERS, colors=event_colors_31)
    + branching_figure(sample, 15, colors=event_colors_31, size=250)
)

## Even the transition graph is tangled

In [ ]:
#| eval: false
taskboard.transition_graph()

In [ ]:
#| echo: false
stage_graph(taskboard, view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

## What kind of visit was it?

In [ ]:
#| echo: false
import json

from taskboard import CHECKOUT, EVERYDAY

# The names the next slides give the clusters (checked against the profiles).
VISITS = ["sync", "focus", "checkout", "review", "checkout_failed", "quick", "onboarding", "planning"]
VISIT_NAMES = {f"cluster_{i}": name for i, name in enumerate(VISITS)}

clusters = taskboard.cluster_analysis(
    features=[{"metric": "event_count_bulk", "metric_args": {"events": EVERYDAY + CHECKOUT}}],
    method_args={"n_clusters": 8},
    scaler="std",
    path_col="session_id",
    overview_metrics=[{"metric": "length"}, {"metric": "event_count_bulk"}],
)
clusters.cluster_renames = json.dumps(VISIT_NAMES)
clusters.slide_height = 600  # the whole slide below the title
clusters


- `sync`: comments, mentions, assignments: working it out with the team
- `focus`: long working visits: tasks created, edited, completed
- `checkout`: everyday work, then a checkout that goes through
- `review`: reports and exports
- `checkout_failed`: a payment error and a support ticket
- `quick`: short visits: notifications, a task or two
- `onboarding`: signup, the tour, a first project
- `planning`: tasks with due dates and dependencies, a look at the timeline


In [ ]:
#| echo: false
#| output: false
visits = (
    taskboard
        .add_clusters(
            "visit", features=[{"metric": "event_count_bulk", "metric_args": {"events": EVERYDAY + CHECKOUT}}],
            method_args={"n_clusters": 8}, scaler="std", path_col="session_id",
        )
        .collapse_events(group_col="session_id", name={"col": "visit"})
        .rename_events(VISIT_NAMES)
)
# Users by the kinds of visits they make (the "Who are the users?" slide).
USER_TYPES = ["paid_teams", "together", "solo_focused", "solo_light", "dropped_off", "invited", "planners"]
USER_NAMES = {f"cluster_{i}": name for i, name in enumerate(USER_TYPES)}
users = visits.add_clusters(
    "user_type", features=[{"metric": "event_count_bulk", "metric_args": {"events": VISITS}}],
    method_args={"n_clusters": 7}, scaler="std", path_col="user_id",
).rename_segment_levels("user_type", USER_NAMES)


## Eight visit types instead of 31 events

In [ ]:
#| echo: false
stage_graph(visits, view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

- After `onboarding`: → `quick` 39%, → `focus` 35%; 14% never come back
- `path_start` → `sync` 21%: invited teammates start right in a team visit
- `sync` → `sync` 20%: a team keeps working it out
- `planning` → `planning` 22%: some plan visit after visit
- `checkout_failed` → `checkout` 31%: a failed payment, then another try
- Before: the same paths, event by event


In [ ]:
#| echo: false
stage_graph(taskboard, view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

## The same paths, visit by visit

In [ ]:
#| echo: false
HTML(
    visit_paths_figure(visits, USERS)
    + branching_figure(visits.sample_paths(n=1000, random_state=7), 5, colors=taskboard_colors(visits.df["event"]), size=250)
)

## Who are the users?

In [ ]:
#| echo: false
user_clusters = visits.cluster_analysis(
    features=[{"metric": "event_count_bulk", "metric_args": {"events": VISITS}}],
    method_args={"n_clusters": 7},
    scaler="std",
    path_col="user_id",
    overview_metrics=[{"metric": "length"}, {"metric": "event_count_bulk"}],
)
user_clusters.cluster_renames = json.dumps(USER_NAMES)
user_clusters.slide_height = 600
user_clusters


- `paid_teams`: every kind of visit, a review and a checkout among them
- `together`: many `sync` visits: pairs, and the teammates they work with
- `solo_focused`: work alone, in long `focus` visits
- `solo_light`: work alone, in `quick` visits
- `dropped_off`: `onboarding` and a visit or two, then gone
- `invited`: came by invite: a `sync`, then `quick` looks, if any
- `planners`: `planning` visit after visit


## What it looks like in Retentioneering code

In [ ]:
#| eval: false
#| code-line-numbers: "|6-12|13|14|16-21"
from taskboard import CHECKOUT, EVERYDAY  # no invite events
VISITS = ["sync", "focus", "checkout", "review",
          "checkout_failed", "quick", "onboarding", "planning"]
visits = (
    taskboard
    .add_clusters(
        "visit",
        features=[{"metric": "event_count_bulk",
                   "metric_args": {"events": EVERYDAY + CHECKOUT}}],
        method_args={"n_clusters": 8}, scaler="std",
        path_col="session_id",
    )
    .collapse_events(group_col="session_id", name={"col": "visit"})
    .rename_events({f"cluster_{i}": v for i, v in enumerate(VISITS)})
)
users = visits.add_clusters(
    "user_type",
    features=[{"metric": "event_count_bulk",
               "metric_args": {"events": VISITS}}],
    method_args={"n_clusters": 7}, scaler="std", path_col="user_id",
)


- `add_clusters`: label every visit (`path_col="session_id"`) with its k-means cluster, the ones `cluster_analysis` showed. The features: everyday actions and checkout. Invite checks are left out: they happen in passing, in visits of any kind
- `collapse_events`: replace each visit's events with a single event, named after its cluster
- `rename_events`: give the clusters the names we read off their profiles. Now `visits.transition_graph()` shows the journey, visit by visit
- `add_clusters` again, on the visits, per user (`path_col="user_id"`): users by the kinds of visits they make


## What makes Retentioneering different

- **Built for clickstreams** with many distinct events, where every path is unique
- **Data processors** made for sequences: a line each to clean, cut, group and label paths
- **Visualizations made for paths**: transition graph, step matrix, cluster analysis, to find what a funnel can't see
- **Together they do more:** processors shape the paths, visualizations read them, as with the visit types here

# Part 2: Retentioneering tools

## Sequence log data

Event log table

| user_id | session_id | event | timestamp | country | platform | n_session |
|---|---|---|---|---|---|---|
| user_0042 | sess_0042_1 | home | 2024-03-13 10:21:14 | US | desktop | 1 |
| user_0042 | sess_0042_1 | catalog | 2024-03-13 10:21:28 | US | desktop | 1 |
| user_0042 | sess_0042_1 | product_view | 2024-03-13 10:22:12 | US | desktop | 1 |
| user_0042 | sess_0042_2 | cart | 2024-03-15 19:02:40 | US | mobile | 2 |

- path identifiers: user_id, *session_id*
- event as a string: event
- event time: timestamp
- segments: *country*, *platform*
- miscellaneous: *n_session*

## Eventstream

- Eventstream: the core class – a data container and the API entry point
- A wrap of pandas.DataFrame
- Schema assigns column roles

In [ ]:
import pandas as pd

df = pd.read_csv("events.csv")
df

In [ ]:
#| code-line-numbers: "|2|3|4|5|6"
stream = rete.Eventstream(df, schema={
    "path_cols": ["user_id", "session_id"],
    "event_col": "event",
    "timestamp_col": "timestamp",
    "segment_cols": ["country", "platform"],
    "custom_cols": ["n_session"],
})

## Data accessor: Eventstream.df

In [ ]:
stream.df

- `event_type`: `raw` for your events; synthetic ones like `path_start` get their own type
- `index`: the event's position in its path
- `subindex`: orders event types that share a timestamp

## 2.1 Data processors

## Types of data processors

Data processor is a method that changes Eventstream and returns its modified version

- Data processors
  - Adding events
    - `add_events`
    - `add_start_end_events`
  - Renaming, editing & merging events
    - `rename_events`
    - `edit_events`
    - `collapse_events`
    - `urls_to_events`
  - Filtering, truncating & sampling
    - `drop_events`
    - `filter_events`
    - `filter_paths` 
    - `truncate_paths`
    - `sample_paths`
  - Sessions & lifecycle
    - `split_sessions`
    - `to_daily_states`
  - Segments & clustering
    - `add_segment`
    - `drop_segment`
    - `add_clusters`
    - `rename_segment_levels`

## Ten toy paths

In [ ]:
#| echo: false
from IPython.display import HTML
from toy import (
    edge_weights_figure,
    grouped_paths_figure,
    metrics_figure,
    pairs_figure,
    paths_figure,
    toy_events,
)

stream = rete.Eventstream(toy_events(), schema={"segment_cols": ["group"]})
HTML(grouped_paths_figure(stream, "group"))

## collapse_events: one event for a run

In [ ]:
one_b = stream.collapse_events(loops=True)

In [ ]:
#| echo: false
HTML(paths_figure(stream, one_b))

- `loops=True`: a run of the same event becomes one: B, B → B
- `group_col`: a run of rows sharing a value becomes one event; that is how visits became events in Part 1
- `event_groups`, `bounds`: a run drawn from a set of events, or everything between two events

## Path metrics: one number per path

In [ ]:
metrics = stream.get_metrics([
    {"metric": "length"},
    {"metric": "has_event", "metric_args": {"event": "X"}},
    {"metric": "matches_pattern", "metric_args": {"pattern": "A->.*->B->.*->C"}},
])

In [ ]:
#| echo: false
HTML(metrics_figure(stream, metrics))

## Path patterns: regex over events

### Syntax

- `->`: then
- `.`: any one event
- `.*`: any run of events, including none
- `[A|D]`: one event, A or D
- `[^X]`: one event, anything but X
- `[^X]*`: a run of events with no X
- `path_start`, `path_end`: the path's own boundaries

### Examples

- `A->B`: A, then B right after it
- `A->.*->C`: A, then C at some point later
- `[A|D]->B`: A or D, then B
- `A->[^X]*->C`: A, then C, with no X in between
- `path_start->[^A|D]`: paths that begin with neither A nor D
- `C->path_end`: paths that end with C

Patterns aren't tied to a path's start or end: `A->.*->C` also matches B→A→C. To pin the start, write `path_start->A->.*->C`.

## truncate_paths: cut a window

In [ ]:
b_to_c = stream.truncate_paths(start_anchor="B", end_anchor="C")

In [ ]:
#| echo: false
HTML(paths_figure(stream, b_to_c, strike=True))

## Anchors: which B, and how far

In [ ]:
last_b = {"pattern": "B", "occurrence": "last"}
after_last_b = stream.truncate_paths(
    start_anchor=last_b,
    end_anchor={**last_b, "offset": 1},
)

In [ ]:
#| echo: false
HTML(paths_figure(stream, after_last_b, strike=True))

## Anchors: a list means whichever comes first

In [ ]:
b_to_c_or_end = stream.truncate_paths(start_anchor="B", end_anchor=["C", "path_end"])

In [ ]:
#| echo: false
HTML(paths_figure(stream, b_to_c_or_end, strike=True))

## add_events: mark a position

In [ ]:
marked = stream.add_events("BC", anchor={"pattern": "B->C", "at": "end"})

In [ ]:
#| echo: false
HTML(paths_figure(marked))

- A synthetic event `BC` where B is followed by C: now a step matrix can center on it, and a funnel can count it
- Other ways to add events: copy existing ones (`source_event`), any SQL (`sql`), or mark churn after days of inactivity (`churn`)

## Chaining data processors

In [ ]:
#| code-line-numbers: "|3|4|5"
clean = (
    stream
        .rename_events({"E": "C"})
        .drop_events(["X"])
        .filter_paths({"metric": "length", "op": ">=", "value": 3})
)

In [ ]:
#| echo: false
from toy import chain_steps_figure

HTML(chain_steps_figure(stream, clean))

## 2.2 Widgets: Funnel vs. Transition Graph

## Funnels: the traditional view

In [ ]:
#| output: false
stream.funnel(steps=["A", "B", "C"])

In [ ]:
#| echo: false
from toy import funnel_figure

HTML(funnel_figure(stream, ["A", "B", "C"]))

What a funnel can't see:

- Other routes: the 30% outside the funnel are a blank; their D→B→E is invisible
- Cycles: A→B→A→B→C counts the same as A→B→C
- Loops: A→B→B→C repeats B; the B→B step is invisible
- Detours: the X in A→X→B→C is invisible
- Other orders: B→A→C stops at A; the B before it is ignored

## Transition graph: count adjacent pairs

Every pair of neighboring events in every path, counted:

In [ ]:
#| echo: false
HTML(pairs_figure(stream))

In [ ]:
#| echo: false
stream.transition_graph(
    edge_weight="count",
    sidebar_open=False,
    view={"edgeFilter": {"mode": "range", "range": [0, 1]}},
)

## Edge weights

In [ ]:
#| echo: false
HTML(edge_weights_figure(
    stream,
    edges=[("A", "B"), ("B", "C"), ("B", "B"), ("B", "E")],
    weights=["count", "unique_paths", "proba_out", "proba_in"],
))

- `count`: how many times the transition happened
- `unique_paths`: how many paths have it; A→B→A→B→C counts once
- `proba_out`: its share of everything that follows the source
- `proba_in`: its share of everything that leads into the target
- Also: `share_of_total`, `avg_per_path`, `time_median`, `time_q95`

## Diff mode: two segments, one graph

In [ ]:
#| echo: false
stage_graph(visits, diff=("channel", "invite", "organic"), view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

- `path_start` → `sync` +69 pp, → `onboarding` −99 pp: invited users start working with the team right away; organic ones start with onboarding
- `transition_graph(diff=("channel", "invite", "organic"))`: each edge is colored by how much more often one segment takes it than the other


## Interactive features (live demo)

- Views
- Search event
- Edge filter
- Node focus + 2-3 clicks
- Selecting traces
- Ego view
- user/session path selector

In [ ]:
#| output: false
ecom = rete.datasets.load_ecom()
checkout = ["cart", "shipping_details", "payment_details", "purchase"]
ecom.transition_graph(views=[
    {"name": "Checkout", "focus": {"type": "path", "nodes": checkout}},
    {"name": "Payment errors",
     "focus": {"type": "node", "event": "payment_error", "direction": "both"}},
])

## 2.3 Widgets: Tree vs. Step matrix 

## Step matrix: count events at each step

In [ ]:
#| echo: false
from toy import prefix_tree_svg, step_layout, step_matrix_table, step_paths_table

HTML(step_layout(
    step_paths_table(stream, 6, spotlight=(1, 2, 3)),
    step_matrix_table(stream, 6, spotlight=(1, 2, 3)),
))

## Tree vs. step matrix

In [ ]:
#| echo: false
HTML(step_layout(
    prefix_tree_svg(stream, 6, spotlight=(1, 2, 3)),
    step_matrix_table(stream, 6, spotlight=(1, 2, 3)),
))

## Step matrix centered on an event

In [ ]:
#| echo: false
HTML(step_layout(
    step_paths_table(stream, 3, spotlight=(0, -1, 1), pattern="B"),
    step_matrix_table(stream, 3, spotlight=(0, -1, 1), pattern="B"),
))

## Why not tree-based diagrams?

In [ ]:
#| echo: false
HTML(branching_figure(
    visits.sample_paths(n=1000, random_state=7), 5,
    colors=taskboard_colors(visits.df["event"]),
))

> A tree branches on every distinct beginning. On real data, almost every path gets a branch of its own.

## Step matrix: one row per visit type (live demo)

The same 20,013 users, visit by visit: one row per visit type, one column per step.

In [ ]:
visits.step_matrix()

## Step matrix: diff mode

In [ ]:
#| eval: false
visits.step_matrix(diff=("channel", "invite", "organic"))

In [ ]:
#| echo: false
diff = visits.step_matrix(diff=("channel", "invite", "organic"))
diff.slide_height = 330
diff

- Each cell: the share in one segment minus the share in the other, step by step
- Invited users are in `sync` (69%) or `quick` (29%) at step 1; organic ones go through `onboarding`, then `focus` and `quick`


## 2.4 Local MCP server

## Start a local server

In [ ]:
#| eval: false
rete.mcp.serve(visits, context={
    "description": "Taskboard, a task tracker for teams. ab_group: the Timeline A/B test",
    "events": {"planning": "a visit spent planning: tasks, due dates, dependencies"},
}, port=8765)


```bash
claude mcp add --transport sse retentioneering http://localhost:8765/sse
```

- The server runs in your Jupyter kernel: the data stay on your machine
- Any MCP agent connects over SSE: Claude Code, Codex, Cursor
- The agent works through tools: `describe`, `add_transition_graph`, `add_step_matrix`, `get_conversion_rate`, `export_report`

## Ask, and get a report

> Why didn't the Timeline test move team activation? Compare test and control, and write a report.

```text
describe()
add_transition_graph(label="Visit flow")
add_transition_graph(label="Test vs control", diff=["ab_group", "test", "control"])
check_analysis(analysis="...")
export_report(title="Why Timeline didn't move team activation", analysis="...")
```

- In the analysis, every number comes with a link to where it was read: `[Test vs control:focus->planning]`

## Every number is a link

In [ ]:
#| echo: false
from retentioneering.mcp import ReportSession, tools

ANALYSIS = """
## The road to a team didn't change

[Test vs control:focus->sync] is the same in both groups: the prompt to plan
with the team brought no more teams in.

## Planning did

In test, more visits lead to planning: [Test vs control:onboarding->planning]
`+3.5 pp` and [Test vs control:focus->planning] `+2.0 pp`. And planning holds on:
[Test vs control:planning->planning] `+10.3 pp`, while overall
[Visit flow:planning->planning] `22%` of planning visits are followed by another one.

## So

Timeline found its users, but they plan alone. Team activation, the test's metric,
can't see them: look at who plans, not at who invites.
"""

session = ReportSession(visits)
_ = tools.add_transition_graph(session, "Visit flow")
_ = tools.add_transition_graph(session, "Test vs control", diff=["ab_group", "test", "control"])
_ = session.export("Why Timeline didn't move team activation", ANALYSIS, "timeline_report.html")
HTML('<iframe class="report-frame" src="timeline_report.html"></iframe>')


<small>Click a link: the report opens its tab and focuses that edge.</small>

# Part 3: A grey A/B test

## The Timeline experiment

- **Pricing:** a workspace is free for up to two members; the Team plan, bought by the owner, adds more
- **Feature:** Timeline, a Gantt-style view of a project. It has always been there as a second tab, and few open it
- **Test:** a new project opens on Timeline, with a prompt: *Plan with your team*
- **Hypothesis:** on a timeline every task wants a date and an assignee; an empty assignee slot (+) invites a teammate right there. Plan the work, and you bring in the people to do it
- **Primary metric:** team activation, the share of new owners whose teammate joins (`teammate_joined`): expected 30% → 33%
- **Design:** every new workspace over six weeks, 50/50: 14,000 owners, and the 6,013 teammates they invited


In [ ]:
#| echo: false
from toy import timeline_mockups

HTML(timeline_mockups())

## The result: grey

In [ ]:
#| echo: false
from toy import ab_results_table

# Workspace owners: the test's unit. Invited teammates never sign up or pay.
owners = taskboard.filter_paths({"op": "=", "metric": "in_segment", "value": 0,
                                 "metric_args": {"segment_name": "channel", "segment_level": "invite"}})
HTML(ab_results_table(owners, {
    "create_project": "Created a project",
    "invite_member": "Invited a teammate",
    "teammate_joined": "Team activation",
    "upgrade": "Upgrade",
}, primary="teammate_joined"))


> Nothing moved. Ship it, or roll it back?

## The funnel: grey at every step

In [ ]:
owners.funnel(
    steps=["signup", "create_project", "invite_member", "teammate_joined", "upgrade"],
    diff=("ab_group", "test", "control"),
)

<small>`owners`: the workspace owners, as in the table before; invited teammates have no `signup`. A funnel counts a user at a step only after every step before it, in order.</small>


## Through the funnel, and back

In [ ]:
#| echo: false
stage_graph(visits, view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

In [ ]:
#| echo: false
back_to_solo = visits.step_matrix(path_pattern="sync", max_steps=4)
back_to_solo.slide_height = 250
back_to_solo


- `sync` → `quick` 38%, → `focus` 19%: after a team visit, the next one is often a short or a lone one
- Step 0 is a user's first team visit. Two visits later only 16% are in a team visit again; 33% are in quick visits, 18% work alone, 18% are gone
- The funnel counted every one of them at `teammate_joined`, as activated


## Who is Timeline for?

In [ ]:
#| echo: false
users_ab = visits.cluster_analysis(
    features=[{"metric": "event_count_bulk", "metric_args": {"events": VISITS}}],
    method_args={"n_clusters": 7},
    scaler="std",
    path_col="user_id",
    overview_metrics=[
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "ab_group"}},
    ],
)
users_ab.cluster_renames = json.dumps(USER_NAMES)
users_ab.slide_height = 260  # a short table: two rows per group
users_ab


- `paid_teams`: owners who grew a team past two, and pay
- `together`: pairs, and the teammates they work with
- `solo_focused`: work alone, in long visits
- `solo_light`: work alone, in short visits
- `dropped_off`: left after a visit or two
- `invited`: came by invite, and mostly look in
- `planners`: plan visit after visit


## Timeline works, for planners

| | `solo_focused`, `solo_light` | `planners` |
|---|---|---|
| Who they are | want a team: invite, wait | freelancers, consultants: plan client work alone |
| Invited a teammate | 79% | 19% |
| Planning, of all visits | 5% | 48% |
| Upgrade | 0% | 0.1% |
| Test users | 50% | 73% |

- Timeline didn't bring teams in: planners don't invite, so team activation couldn't move
- It kept planners: in control most of them never find it, and drop off
- Team sells seats past two, and planners have no one to seat. Next: show Timeline to those who plan, and test a solo Pro plan for them


## Digging deeper: a loop on the timeline

In [ ]:
#| echo: false
user_type = users.df.groupby("user_id")["user_type"].first()
planners = (
    taskboard
        .add_segment("user_type", func=lambda df: df["user_id"].map(user_type))
        .filter_paths({"op": "=", "metric": "in_segment", "value": 1,
                       "metric_args": {"segment_name": "user_type", "segment_level": "planners"}})
)
stage_graph(planners, view={"edgeFilter": {"mode": "range", "range": [0, 1]}})


- `view_timeline` → `view_timeline` 19%: planners open the timeline, and open it again, in the same visit
- 30% of planners have this loop. They make 7 visits instead of 11, and stay 15 days instead of 23
- What are they doing there? The log can't say: nothing they do on that screen is tracked


## Digging deeper: what they click

- **Ask:** UX interviews and usability sessions with planners, at work on the timeline
- **Track:** every click on the timeline screen, with its coordinates
- **Find:** users grab the ends of the task bars, to move a start or stretch a deadline. The bars don't move; users open the timeline again
- **Fix:** drag a bar's end to change its dates; click the bar to open the task

<small>A real case from Maxim Godzi's practice, replayed on Taskboard: users dropped off over UI elements they took for buttons. A loop on the transition graph gave it away; click coordinates showed where.</small>

In [ ]:
#| echo: false
from toy import timeline_clicks_mockups

HTML(timeline_clicks_mockups())

## Where we could have stopped

| We could have stopped at | Digging further, we found | With |
|---|---|---|
| A grey test: roll Timeline back | Teams form and fall apart, and the funnel counts them as activated | Transition graph, step matrix |
| "Timeline doesn't work" | A cluster of users, 73% of them in test: planners, who work alone | Cluster analysis, `in_segment_bulk` |
| "It works for planners" | A loop on the timeline: bars that look draggable, but aren't. Fixed | Transition graph, then UX interviews and click tracking |

> From a grey number down to what users actually do, and to a better product